In [ ]:
import re
import sys
from pathlib import Path

HICOLA_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'hicola_paths.py').exists())
sys.path.insert(0, str(HICOLA_ROOT / 'scripts'))
import hicola_paths as hp
import analysis as an

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

%config InlineBackend.figure_formats = ['retina']   # PNG at twice the resolution: sharp and copyable

# Resolution tests on Abacus_Small with matched initial conditions (LCDM, z = 0.5)

<!-- The same kind of tests as `3test.ipynb`, now with the initial conditions of the Abacus box itself:
the white noise of AbacusSummit `small_c000_ph3000` (regenerated at 1024³ with zeldovich-PLT,
`jobs/white_noise_abacus.sh`) instead of a random draw (`ic_random_field_type = "read_white_noise"`,
realization `ph3000`), scaled with Abacus's own initial P(k) given at z = 99
(`abacus_cosm000_ic_pk_z99.000.dat`, `ic_input_redshift = 99`). The runs therefore share the Abacus box's
phases and realized amplitudes and compare with it directly.

Four runs, z_ini = 19 and P(k) by PCS in all of them: 50 and 30 time steps, each with force mesh 3N and 2N
(the P(k) measurement mesh `pofk_nmesh` follows the force mesh). Every run is the corresponding
`LCDM_steps<N>_zini19_mesh<m>_s1234.lua` of `3test.ipynb` with only the initial-condition settings
changed (500 Mpc/h, 1024³ particles = N³, linear time steps in a, a single output at z = 0.5).

P(k) is Hi-COLA's own measurement on the snapshot (`pofk_z0.500.txt`), the HMF is from
`jobs/halo_finder.sh` (FoF b = 0.2, at least 50 particles). Settings and costs are read from each
run's backend job log in `HiCOLA/Output/logs/`. -->

In [ ]:
target = 'Abacus_Small'
redshift = 0.5               # the only output of the test runs
realization = 'ph3000'       # the white noise of AbacusSummit small_c000_ph3000
reference_name = 'Nbody'
reference_boxes = [1]        # Abacus_Small has one box: small_c000_ph3000 itself
# Same k range as compare_Abacus_Small (the largest k of the COLA_NM4096 reference)
k_max = an.load_table(hp.reference_file('COLA_NM4096', 'LCDM', 1, 1.65, 'PS'))[0].max()
M_min = 1e12                 # lowest halo mass shown
LOG_DIR = HICOLA_ROOT / 'HiCOLA' / 'Output' / 'logs'

# Backend job of each run (model): its logs LOG_DIR/backend_<id>.out/.err give the settings and costs.
# None until the run is submitted. The first is the base (highest resolution): right panels of the figures.
backend_jobs = {
    'LCDM_steps50_zini19_mesh3': None,
    'LCDM_steps50_zini19_mesh2': None,
    'LCDM_steps30_zini19_mesh3': None,
    'LCDM_steps30_zini19_mesh2': None,
}
models = list(backend_jobs)
base = models[0]
settings = ('steps', 'mesh')   # the settings named in the legends and the table

In [ ]:
# Figure styles (as in compare_Abacus_Small)
S = dict(label=10.5, tick=8.8, legend=7.8, lw=1.45)
REFERENCE_COLOR = '#1f4e79'
RATIO_REF_COLOR = '0.35'
RUN_COLORS = ['#222222', '#c47f2b', '#4c8c4a', '#8e5ea2', '#b5446e', '#2a9d8f', '#d1495b', '#6c757d',
              '#e9c46a', '#264653', '#9c6644', '#00a6ed']
RUN_STYLES = ['-', '--', '-.', ':', (0, (5, 1)), (0, (3, 1, 1, 1, 1, 1)), (0, (1, 1)), (0, (5, 2, 1, 2)),
              (0, (1, 2)), (0, (4, 1, 1, 1)), (0, (6, 1, 2, 1)), (0, (2, 2))]   # 12 distinct runs
K_LABEL = r'$k\,[h\,\mathrm{Mpc}^{-1}]$'
M_LABEL = r'$M\,[h^{-1}M_\odot]$'


def run_style(i):
    return dict(color=RUN_COLORS[i % len(RUN_COLORS)], ls=RUN_STYLES[i % len(RUN_STYLES)])


def finish(axes):
    for ax in np.atleast_1d(axes).ravel():
        ax.minorticks_on()
        ax.grid(True, which='major', ls=':', color='0.82', alpha=0.75, lw=0.45)
        ax.tick_params(axis='both', which='major', direction='in', top=True, right=True, labelsize=S['tick'], length=3.4, width=0.7)
        ax.tick_params(axis='both', which='minor', direction='in', top=True, right=True, labelsize=S['tick'] - 0.8, length=2.0, width=0.55)
        for spine in ax.spines.values():
            spine.set_linewidth(0.7)


def legend(ax, **kwargs):
    ax.legend(fontsize=S['legend'], frameon=False, handlelength=1.8, borderaxespad=0.3, labelspacing=0.3, **kwargs)

## Load

In [ ]:
def read_log(model):
    """Settings and costs of a run from its backend job logs (None where the job has not written them yet)."""
    job = backend_jobs[model]
    if job is None:
        return None
    out_file, err_file = LOG_DIR / f'backend_{job}.out', LOG_DIR / f'backend_{job}.err'
    if not out_file.exists():
        return None
    out = out_file.read_text()
    err = err_file.read_text() if err_file.exists() else ''

    def param(key):
        found = re.search(rf'^\s*{key}\s*:\s*(.*?)\s*$', out, re.M)   # parameters as nbody prints them
        return found.group(1).strip('[] ') if found else None

    name = param('simulation_name')
    if name != hp.simulation_name(target, model, realization):
        raise ValueError(f'{out_file} is the log of {name}, not of {model}_{realization}')
    npart = int(param('particle_Npart_1D'))
    resources = re.search(r'^Resources:\s+(\d+) nodes, (\d+) MPI tasks', out, re.M)
    rss = re.search(r'\[Peak resident set size\]:\s*\nMin over tasks:\s*([\d.]+)\s*MB\s*\n'
                    r'Mean over tasks:\s*([\d.]+)\s*MB\s*\nMax over tasks:\s*([\d.]+)\s*MB', out)
    real = re.search(r'^real\s+(\d+)m([\d.]+)s', err, re.M)   # wall time of nbody (time mpirun ...)
    nodes, tasks = (int(resources.group(1)), int(resources.group(2))) if resources else (None, None)
    hours = (int(real.group(1)) * 60 + float(real.group(2))) / 3600 if real else None
    return {
        'job': job,
        'steps': int(param('timestep_nsteps')),
        'spacing': param('timestep_scalefactor_spacing'),
        'z_ini': float(param('ic_initial_redshift')),
        'force mesh': int(param('force_nmesh')) / npart,
        'P(k) mesh': int(param('pofk_nmesh')) / npart,
        'outputs z': param('output_redshifts'),
        'wall time [h]': hours,
        'nodes': nodes,
        'MPI tasks': tasks,
        'node hours': nodes * hours if hours is not None else None,
        'peak RSS per task, max [GB]': float(rss.group(3)) / 1024 if rss else None,
        'peak RSS, sum over tasks [GB]': float(rss.group(2)) * tasks / 1024 if rss else None,
    }


def run_info(run):
    """Settings and costs of a run (None until its backend job has written its log)."""
    return logs.get(run)


def run_label(run, settings=('steps', 'mesh'), latex=True):
    """Name of a run, naming only the given settings ('mesh' adds the P(k) mesh when it differs)."""
    info = run_info(run)
    if info is None:
        return run
    parts = []
    if 'steps' in settings:
        parts.append(f"{info['steps']} steps")
    if 'z_ini' in settings:
        parts.append(f"$z_\\mathrm{{ini}}$ = {info['z_ini']:g}" if latex else f"z_ini = {info['z_ini']:g}")
    if 'mesh' in settings:
        parts.append(f"force mesh {info['force mesh']:g}N")
        if info['P(k) mesh'] != info['force mesh']:
            parts.append(f"P(k) mesh {info['P(k) mesh']:g}N")
    return ', '.join(parts)


logs = {model: read_log(model) for model in backend_jobs}
for model, info in logs.items():
    if info is None or info['wall time [h]'] is None:
        print(f'{model}/{realization}: job {backend_jobs[model]} not finished (or not started)')

# Reference (Abacus, LCDM) and Hi-COLA measurements; runs without a measurement (yet) are skipped
ref = {kind: an.load_reference(target, 'LCDM', redshift, kind, boxes=reference_boxes)[:2] for kind in ('PS', 'HMF')}
hic = {}
for kind in ('PS', 'HMF'):
    for model in models:
        try:
            hic[kind, model] = an.load_hicola(target, model, [realization], redshift, kind)
        except FileNotFoundError as error:
            print(f'Skipping {model} / {kind}: {error}')


## Functions

<!-- As in `3test.ipynb`. The P(k) values in the table are at single k: Hi-COLA's P(k) is interpolated
linearly (on the Abacus k bins for P/P_Nbody, then in log k) between its fine k bins of width 2π/L. -->

In [ ]:
PERCENT = FuncFormatter(lambda v, _: f'{v:g}%')   # tick labels of values already in %, e.g. -2%, 0.5%


def plot_pk(models, base, settings, shown=None):
    """P(k) of the runs: with the Abacus box, ratio to it, ratio to the run `base`. Legends name `settings`.
    Only the runs `shown` (default all) are in the left and middle panels; the others get a legend on the right."""
    styles = {model: run_style(i) for i, model in enumerate(models)}
    shown = models if shown is None else shown
    fig, axes = plt.subplots(1, 3, figsize=(11, 2.9), dpi=300, constrained_layout=True)
    ax_p, ax_pr, ax_pb = axes
    k_ref, P_ref = ref['PS']
    keep_ref = an.within(k_ref, None, k_max) & (P_ref > 0)   # few-mode Abacus bins can be <= 0 after shot noise
    ax_p.semilogx(k_ref[keep_ref], k_ref[keep_ref]**1.5 * P_ref[keep_ref], lw=S['lw'], color=REFERENCE_COLOR, label=reference_name)
    for model in models:
        if ('PS', model) not in hic:
            continue
        k, P = hic['PS', model]
        keep = an.within(k, k_ref.min(), k_max)
        if model in shown:
            ax_p.semilogx(k[keep], k[keep]**1.5 * P[keep], lw=S['lw'], label=run_label(model, settings), **styles[model])
            ax_pr.semilogx(k_ref[keep_ref], 100 * (an.ratio(k, P, k_ref, P_ref)[keep_ref] - 1), lw=S['lw'], **styles[model])
        if ('PS', base) in hic:
            k_b, P_b = hic['PS', base]
            keep_b = an.within(k_b, k_ref.min(), k_max)
            ax_pb.semilogx(k_b[keep_b], 100 * (an.ratio(k, P, k_b, P_b)[keep_b] - 1), lw=S['lw'], **styles[model],
                           label=None if model in shown else run_label(model, settings))
    ax_p.set_ylabel(r'$k^{3/2} P(k)$', fontsize=S['label'])
    ax_pr.set_ylabel(rf'$P / P_\mathrm{{{reference_name}}} - 1$', fontsize=S['label'])
    ax_pb.set_ylabel(r'$P / P_\mathrm{base} - 1$', fontsize=S['label'])
    for ax in axes:
        ax.set_xlabel(K_LABEL, fontsize=S['label'])
    ax_pr.axhline(0.0, color=RATIO_REF_COLOR, lw=0.9, ls=':')
    ax_pb.axhline(0.0, color=RATIO_REF_COLOR, lw=0.9, ls=':')
    legend(ax_p)
    if set(models) - set(shown):
        legend(ax_pb)
    finish(axes)
    for ax in (ax_pr, ax_pb):
        ax.yaxis.set_major_formatter(PERCENT)
    plt.show()


def plot_hmf(models, base, settings, shown=None):
    """Cumulative HMF of the runs: with the Abacus box, ratio to it, ratio to the run `base`. Legends name `settings`.
    Only the runs `shown` (default all) are in the left and middle panels; the others get a legend on the right."""
    styles = {model: run_style(i) for i, model in enumerate(models)}
    shown = models if shown is None else shown
    fig, axes = plt.subplots(1, 3, figsize=(11, 2.9), dpi=300, constrained_layout=True)
    ax_n, ax_nr, ax_nb = axes
    M_ref, n_ref = ref['HMF']
    keep_ref = (n_ref > 0) & (M_ref >= M_min)
    ax_n.loglog(M_ref[keep_ref], n_ref[keep_ref], lw=S['lw'], color=REFERENCE_COLOR, label=reference_name)
    for model in models:
        if ('HMF', model) not in hic:
            continue
        M, n = hic['HMF', model]
        keep = (n > 0) & (M >= M_min)
        if model in shown:
            ax_n.loglog(M[keep], n[keep], lw=S['lw'], label=run_label(model, settings), **styles[model])
            r = an.ratio(M, n, M_ref, n_ref)
            keep_r = np.isfinite(r) & (M_ref >= M_min)
            ax_nr.semilogx(M_ref[keep_r], 100 * (r[keep_r] - 1), lw=S['lw'], **styles[model])
        if ('HMF', base) in hic:
            M_b, n_b = hic['HMF', base]
            r = an.ratio(M, n, M_b, n_b)
            keep_b = np.isfinite(r) & (M_b >= M_min)
            ax_nb.semilogx(M_b[keep_b], 100 * (r[keep_b] - 1), lw=S['lw'], **styles[model],
                           label=None if model in shown else run_label(model, settings))
    ax_n.set_ylabel(r'$n(>M)\,[h^3\,\mathrm{Mpc}^{-3}]$', fontsize=S['label'])
    ax_nr.set_ylabel(rf'$n / n_\mathrm{{{reference_name}}} - 1$', fontsize=S['label'])
    ax_nb.set_ylabel(r'$n / n_\mathrm{base} - 1$', fontsize=S['label'])
    for ax in axes:
        ax.set_xlabel(M_LABEL, fontsize=S['label'])
    ax_nr.axhline(0.0, color=RATIO_REF_COLOR, lw=0.9, ls=':')
    ax_nb.axhline(0.0, color=RATIO_REF_COLOR, lw=0.9, ls=':')
    legend(ax_n)
    if set(models) - set(shown):
        legend(ax_nb)
    finish(axes)
    for ax in (ax_nr, ax_nb):
        ax.yaxis.set_major_formatter(PERCENT)
    plt.show()


K_TABLE = (0.3, 0.7, 1.0)   # h/Mpc, where the table gives the P(k) deviation from the reference


def pk_deviation(model, k_at, against=None):
    """100 (P/P_ref - 1) of a run at k_at: the ratio on the reference k bins, interpolated linearly in log k.
    With `against` (a run), the ratio to that run on its own k bins instead (0 for that run itself)."""
    if ('PS', model) not in hic or (against is not None and ('PS', against) not in hic):
        return np.nan
    k_ref, P_ref = ref['PS'] if against is None else hic['PS', against]
    r = an.ratio(*hic['PS', model], k_ref, P_ref)
    good = np.isfinite(r)
    return 100 * (np.interp(np.log(k_at), np.log(k_ref[good]), r[good]) - 1)


def cost_table(models, base, settings, relative=False, pk=True, reference_row=None, costs=True, nbody_row=False):
    """Costs of the runs as a table figure (copyable like the plots), rows named by `settings`, the run `base` marked.
    P(k) columns: P/P_Nbody - 1; with `relative`, P/P_base - 1, except in the row of the run `reference_row`
    (first row, set off by a dashed rule), which keeps P/P_Nbody - 1; none with pk=False (runs without a reference, e.g. nDGP).
    costs=False leaves out the cost columns (e.g. test 5, where all rows are the same simulation).
    nbody_row=True adds a first row for the reference itself (Nbody: P(k) columns 0, no costs)."""
    rows = {run: run_info(run) for run in models if run_info(run) and run_info(run)['wall time [h]'] is not None}
    if nbody_row:   # the reference itself, first
        rows = {reference_name: None, **rows}
    if relative:   # header shared by the K_TABLE columns
        deviation = r'P/P$_\mathrm{base}$ - 1' + (rf'   (paired-fixed row: P/P$_\mathrm{{{reference_name}}}$ - 1)'
                                                    if reference_row else '')
    else:
        deviation = rf'P/P$_\mathrm{{{reference_name}}}$ - 1'

    def against(m):   # denominator of the P(k) columns: None = the reference (Abacus)
        return base if relative and m != reference_row else None

    columns = [   # (shared header or None, header, value of a run, format, width [in])
        *[(None, 'nbody wall\ntime [h]', lambda m: rows[m]['wall time [h]'], '{:.2f}', 1.4),
          (None, 'node\nhours', lambda m: rows[m]['node hours'], '{:.1f}', 1.1)][:2 if costs else 0],
        *[(deviation, f'k = {k_table:g}', lambda m, k_table=k_table: round(pk_deviation(m, k_table, against(m)), 1) + 0.0, '{:+.1f}%',   # + 0.0: no '-0.0'
           1.3 if relative else 0.95) for k_table in K_TABLE if pk],
    ]
    resources = {(rows[m]['nodes'], rows[m]['MPI tasks']) for m in rows if rows[m] is not None}
    if not costs:
        captions = []
    elif len(resources) == 1:
        nodes, tasks = next(iter(resources))
        captions = [f'{nodes} nodes, {tasks} MPI tasks each']
    else:   # different resources: show them per run
        columns = [(None, 'nodes x\ntasks', lambda m: f"{rows[m]['nodes']} x {rows[m]['MPI tasks']}", '{}', 1.1), *columns]
        captions = []
    def note(m):   # base mark, except on a run of several realizations (the pair: its costs are summed over them)
        return ' (base)' if m == base else ''

    labels = [m if rows[m] is None else run_label(m, settings, latex=False) + note(m) for m in rows]

    # Layout in inches: a label column, then the value columns (right-aligned at x_right); booktabs-style rules
    size, row_h, head_h, pad, line = 9, 0.25, 0.44, 0.12, 0.09   # line: half the spacing of the two header lines
    label_w = 0.2 + 0.062 * max(len(label) for label in labels + ['run'])
    x_right = pad + label_w + np.cumsum([width for *_, width in columns]) - 0.08
    width = pad + label_w + sum(width for *_, width in columns) + pad
    height = pad + head_h + row_h * len(rows) + 0.1 + 0.18 * len(captions) + pad
    fig = plt.figure(figsize=(width, height), dpi=300)
    fig.add_axes([0, 0, 1, 1]).axis('off')   # the inline display skips figures without axes
    inch = fig.dpi_scale_trans
    top = height - pad
    y_head = top - head_h / 2

    def rule(y, lw, x0=pad, x1=width - pad):
        fig.add_artist(plt.Line2D([x0, x1], [y, y], transform=inch, color='black', lw=lw))

    def text(x, y, s, ha='right', **kw):
        fig.text(x, y, s, transform=inch, fontsize=size, va='center', ha=ha, **kw)

    rule(top, 1.0)
    text(pad + 0.05, y_head, 'run', ha='left', weight='bold')
    for j, (shared, header, *_) in enumerate(columns):
        if shared is None:
            text(x_right[j], y_head, header, weight='bold', linespacing=1.15)
        else:   # lower line; the shared header goes once, centred above its columns
            text(x_right[j], y_head - line, header, weight='bold')
    for shared in dict.fromkeys(col[0] for col in columns if col[0] is not None):
        js = [j for j, col in enumerate(columns) if col[0] == shared]
        x0, x1 = x_right[js[0]] - columns[js[0]][4] + 0.3, x_right[js[-1]]
        text((x0 + x1) / 2, y_head + line, shared, ha='center', weight='bold')
    rule(top - head_h, 0.6)
    for i, (model, label) in enumerate(zip(rows, labels)):
        y = top - head_h - row_h * (i + 0.5)
        text(pad + 0.05, y, label, ha='left')
        for j, (shared, _, value, fmt, _) in enumerate(columns):
            if rows[model] is None:   # the reference row: 0 in the P(k) columns, nothing else
                text(x_right[j], y, fmt.format(0.0) if shared is not None else '–')
            else:
                text(x_right[j], y, fmt.format(value(model)))
        if model == reference_row:   # dashed rule under the reference row: its values mean something else
            fig.add_artist(plt.Line2D([pad, width - pad], [y - row_h / 2] * 2, transform=inch, color='0.35', lw=0.6, ls='--'))
    bottom = top - head_h - row_h * len(rows)
    rule(bottom, 1.0)
    for i, caption in enumerate(captions):
        fig.text(pad + 0.05, bottom - 0.17 - 0.18 * i, caption, transform=inch, fontsize=size - 1, va='center', ha='left', color='0.3')
    plt.show()


## Time steps (50, 30) × force mesh (3N, 2N), z_ini = 19, PCS

<!-- Left: the measurement with the Abacus box; middle: deviation from the Abacus box (same initial
conditions); right: deviation from the base run, 50 steps with force mesh 3N (black). -->

In [ ]:
plot_pk(models, base, settings)

In [ ]:
plot_hmf(models, base, settings)

In [ ]:
cost_table(models, base, settings, nbody_row=True)